In [ ]:
import requests
import lxml.html
import pandas as pd
import os
import us
import jellyfish

from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional
from google import genai
from pathlib import Path
from google.genai import types
from pypdf import PdfReader
from datetime import datetime

In [4]:
from ingest_sc_cases import case_df

In [5]:
from ingest_sc_judges import judge_pd

In [6]:
# Cases with no opinion documents
case_df[~case_df["opinion_link"].str.contains("statecourt", na=False)]

,docket_no,title,state,date,type,pending,opinion_link
13,2026-00384,Williams v. Board of Elections of the State of...,New York,"March 19, 2026",Voting Rights and Elections,False,None
38,20220896,State v. Andrus,Utah,"August 7, 2025",Criminal Law,False,None
50,S-1-SC-40715,Franklin v. Martinez,New Mexico,"January 26, 2026",Criminal Law,False,None
98,22 OC 00022 1 B,Persaud-Zamora v. Cegavske,Nevada,"April 26, 2022",Voting Rights and Elections,False,None
102,EF2025-2536,Texas v. Bruck,Texas,"October 31, 2025",Reproductive Rights,False,None
104,2025AP002121,Voces de la Frontera v. Gerber,Wisconsin,"September 18, 2025",Government Structure,False,None
121,PR-123179,McVay v. Cockroft,Oklahoma,"July 28, 2025",Voting Rights and Elections,False,None
153,15-25-00093-CV,State v. City of San Antonio,Texas,"June 19, 2025",Reproductive Rights,False,None
207,CV01-23-14744,Adkins v. State,Idaho,"April 11, 2025",Reproductive Rights,False,None
229,SJC-13666,Gotay v. Creen,Massachusetts,"March 21, 2025",Civil Due Process,False,None


In [48]:
class Opinion(BaseModel):
    judge_name: str = Field(description="The full name of the judge giving an opinion.")
    opinion_summary: str = Field(
        description="An extremely brief description of the judge's opinion on the case."
    )


class Politics(BaseModel):
    dim_envir: bool = Field(description="Protects the environment")
    dim_consum: bool = Field(description="Protects consumers")
    dim_reprights: bool = Field(description="Protects reproductive rights")
    dim_demnorms: bool = Field(description="Protects democratic norms")
    dim_press: bool = Field(description="Protects the press")
    dim_pubhealth: bool = Field(description="Protects public health")
    dim_church: bool = Field(description="Protects the separation of church and state")
    dim_voting: bool = Field(description="Protects voting access")
    dim_pubedu: bool = Field(description="Protects public education")
    dim_civlib: bool = Field(description="Protects civil liberties")
    dim_priv: bool = Field(description="Protects privacy")
    dim_workers: bool = Field(description="Protects workers")


class Case(BaseModel):
    title: str = Field(description="The title of the case.")
    decision: str = Field(description="The final decision that was made for the case.")
    issue_debate: str = Field(description="The main issue being debated in the case.")
    plaintiff_issues: Politics
    judge_opinions: List[Opinion]

In [49]:
def read_opinion(pdf_link: str, model_id: str, client: genai.Client, prompt: str):
    resp = requests.get(pdf_link).content

    genai_resp = client.models.generate_content(
        model=model_id,
        contents=[types.Part.from_bytes(data=resp, mime_type="application/pdf"), prompt],
        config={
            "response_mime_type": "application/json",
            "response_json_schema": Case.model_json_schema(),
        },
    )

    return Case.model_validate_json(genai_resp.text)

In [50]:
def analyze_state_cases(
    case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_start: str, client_info: dict
):
    prompt = prompt_start + "$DATAFRAME$" + judge_df.to_markdown(index=False)
    num_cases = len(case_df)
    model_id = client_info["model_id"]
    client = client_info["client"]

    file_dic = {}
    for i in range(num_cases):
        print(f"Querying case {i}")
        docket_no = case_df.iloc[i]["docket_no"].replace(" ", "-")
        state = case_df.iloc[i]["state"]
        date = str(datetime.strptime(case_df.iloc[i]["date"], "%B %d, %Y"))[:10].replace("-", "/")
        pdf_link = case_df.iloc[i]["opinion_link"]

        case_ref = "_".join([docket_no, state, date])

        opinion_resp = read_opinion(pdf_link, model_id, client, prompt)

        file_dic[case_ref] = {}
        file_dic[case_ref]["pdf_link"] = pdf_link
        file_dic[case_ref]["response"] = opinion_resp.model_dump()

    return file_dic

In [51]:
def apply_model(case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_path: str):
    with open(prompt_path, "r") as prompt_file:
        prompt_start = prompt_file.read()

    load_dotenv(find_dotenv())

    gemini_key = os.getenv("GEMINI_API_KEY")
    client_info = {"model_id": "gemini-2.5-flash-lite", "client": genai.Client(api_key=gemini_key)}

    case_dic = analyze_state_cases(case_df, judge_df, prompt_start, client_info)

    return case_dic

In [52]:
def state_opinion_table(case_dic: dict):
    opinion_table = {"case_id": [], "name": [], "opinion": []}

    for key in case_dic.keys():
        opinions = case_dic[key]["response"]
        num_opinions = len(opinions["judge_opinions"])

        for i in range(num_opinions):
            opinion_table["case_id"].append(key)
            judge_name = opinions["judge_opinions"][i]["judge_name"]
            opinion_table["name"].append(judge_name)
            opinion_summary = opinions["judge_opinions"][i]["opinion_summary"]
            opinion_table["opinion"].append(opinion_summary)

    return pd.DataFrame(opinion_table)

In [56]:
def state_case_table(case_df: pd.DataFrame, case_dic: dict):
    case_table = {
        "case_id": [],
        "docket_no": [],
        "title": [],
        "state": [],
        "date": [],
        "type": [],
        "dispute_desc": [],
        "decision_outcome": [],
        "decision_status": [],
        "dim_envir": [],
        "dim_consum": [],
        "dim_reprights": [],
        "dim_demnorms": [],
        "dim_press": [],
        "dim_pubhealth": [],
        "dim_church": [],
        "dim_voting": [],
        "dim_pubedu": [],
        "dim_civlib": [],
        "dim_priv": [],
        "dim_workers": [],
    }
    num_cases = len(case_df)

    for i in range(num_cases):
        docket_no = case_df.iloc[i]["docket_no"].replace(" ", "-")
        case_table["docket_no"].append(docket_no)
        state = case_df.iloc[i]["state"]
        case_table["state"].append(state)
        date = str(datetime.strptime(case_df.iloc[i]["date"], "%B %d, %Y"))[:10].replace("-", "/")
        case_table["date"].append(date)

        title = case_df.iloc[i]["title"]
        case_table["title"].append(title)
        type = case_df.iloc[i]["type"]
        case_table["type"].append(type)

        case_id = "_".join([docket_no, state, date])
        case_table["case_id"].append(case_id)

        status = case_df.iloc[i]["pending"]
        case_table["decision_status"].append(status)

        response = case_dic[case_id]["response"]
        case_table["dispute_desc"].append(response["issue_debate"])
        case_table["decision_outcome"].append(response["decision"])

        politics = response["plaintiff_issues"]
        for key in politics.keys():
            case_table[key].append(politics[key])

    return pd.DataFrame(case_table)

In [54]:
def produce_tables(case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_path: str):
    states = case_df["state"].sort_values().unique()

    opinions = []
    cases = []

    for state in states:
        state_cases = case_df[case_df["state"] == state]

        state_abbr = str(us.states.lookup(state).abbr)
        state_judges = judge_df[judge_df["state"] == state_abbr]

        case_dic = apply_model(state_cases, state_judges, prompt_path)

        opinions.append(state_opinion_table(case_dic))
        cases.append(state_case_table(state_cases, case_dic))

    rd = {"opinion_table": pd.concat(opinions), "case_table": pd.concat(cases)}

    return rd

In [57]:
mon_cases = case_df[case_df["state"] == "Montana"].head(5)
mon_judges = judge_pd[judge_pd["state"] == "MT"]

mon_opinions = produce_tables(mon_cases, mon_judges, "prompt.txt")

Querying case 0
Querying case 1
Querying case 2
Querying case 3
Querying case 4


In [58]:
display(mon_opinions["opinion_table"])
display(mon_opinions["case_table"])

,case_id,name,opinion
0,25-0139_Montana_2026/04/14,Laurie McKinnon,"Delivered the Opinion of the Court, affirming ..."
1,25-0139_Montana_2026/04/14,James Jeremiah Shea,Concurred in the decision.
2,25-0139_Montana_2026/04/14,Ingrid Gustafson,Concurred in the decision.
3,25-0139_Montana_2026/04/14,Katherine Biidegaray,Concurred in the decision.
4,25-0139_Montana_2026/04/14,Beth Baker,"Specially concurred, agreeing to uphold the pr..."
5,25-0139_Montana_2026/04/14,James A. Rice,"Dissented, arguing the decision forces the Sta..."
6,25-0139_Montana_2026/04/14,Cory Swanson,Joined the dissenting opinion and wrote separa...
7,OP-25-0858_Montana_2026/02/27,Katherine Biidegaray,"Delivered the Opinion and Order of the Court, ..."
8,OP-25-0858_Montana_2026/02/27,James Jeremiah Shea,Concurred with the Court's decision.
9,OP-25-0858_Montana_2026/02/27,Laurie McKinnon,Concurred with the Court's decision.


,case_id,docket_no,title,state,date,type,dispute_desc,decision_outcome,decision_status,dim_envir,...,dim_reprights,dim_demnorms,dim_press,dim_pubhealth,dim_church,dim_voting,dim_pubedu,dim_civlib,dim_priv,dim_workers
0,25-0139_Montana_2026/04/14,25-0139,Kalarchik v. State,Montana,2026/04/14,Civil Rights,"The main issue is whether the State Policies, ...",The Supreme Court affirmed the District Court'...,False,False,...,False,False,False,False,False,False,False,True,True,False
1,OP-25-0858_Montana_2026/02/27,OP-25-0858,Kendrick v. Knudsen,Montana,2026/02/27,Voting Rights and Elections,The main issue is whether Ballot Issue 8 (BI-8...,"The petition is granted, the Attorney General'...",False,False,...,False,False,False,False,False,True,False,True,False,False
2,PR-23-0496_Montana_2025/12/31,PR-23-0496,In the Matter of Austin Miles Knudsen,Montana,2025/12/31,Government Structure,Whether Attorney General Knudsen violated the ...,This case is DISMISSED.,False,False,...,False,False,False,False,False,False,False,False,False,False
3,DA-24-0039_Montana_2026/03/17,DA-24-0039,Montanans Against Irresponsible Densification ...,Montana,2026/03/17,Civil Rights,The central debate is whether Montana's Land U...,The Supreme Court reversed the District Court'...,False,False,...,False,False,False,False,False,False,False,True,False,False
4,DA-24-0250_Montana_2025/08/05,DA-24-0250,State v. Alford,Montana,2025/08/05,Criminal Law,The main issue debated is whether the mandator...,The District Court did not err when it sentenc...,False,False,...,False,False,False,True,False,False,False,False,False,False


In [36]:
def find_match(name_str: str, name_list: list):
    sim_scores = {"alias": name_list, "sim_score": []}

    for alias in name_list:
        sim_score = jellyfish.jaro_winkler_similarity(name_str, alias)
        sim_scores["sim_score"].append(sim_score)

    return pd.DataFrame(sim_scores).sort_values(by="sim_score", ascending=False).iloc[0]["alias"]

In [38]:
def standardize_names(opinion_table: pd.DataFrame, judge_df: pd.DataFrame):
    r_table = opinion_table.copy()
    known_names = list(judge_df["name"])

    for i, j_name in enumerate(opinion_table["name"]):
        alias = find_match(j_name, known_names)
        r_table.loc[i, "name"] = alias

    return r_table

In [42]:
standardize_names(mon_opinions["opinion_table"], mon_judges)

,case_id,name,opinion
0,25-0139_Montana_2026/04/14,Laurie McKinnon,"Delivered the opinion of the Court, affirming ..."
1,25-0139_Montana_2026/04/14,James Jeremiah Shea,Concurred with the majority opinion.
2,25-0139_Montana_2026/04/14,Ingrid Gustafson,Concurred with the majority opinion.
3,25-0139_Montana_2026/04/14,Katherine Biidegaray,Concurred with the majority opinion.
4,25-0139_Montana_2026/04/14,Beth Baker,"Specially concurred, upholding the preliminary..."
5,25-0139_Montana_2026/04/14,James A. Rice,"Dissented, arguing the decision forces the sta..."
6,25-0139_Montana_2026/04/14,Cory Swanson,"Joined the dissenting opinion, expressing conc..."
7,OP-25-0858_Montana_2026/02/27,Katherine Biidegaray,"Delivered the Opinion and Order of the Court, ..."
8,OP-25-0858_Montana_2026/02/27,James Jeremiah Shea,Concurred with the Court's decision.
9,OP-25-0858_Montana_2026/02/27,Laurie McKinnon,Concurred with the Court's decision.
